# MalariAI — Phase 3: EfficientNet-B0 + Focal Loss (Kaggle)

**Kaysarul Anas Apurba · Laurentian University**  
Pipeline B · Stage 2 · EfficientNet-B0 · NIH BBBC041 · 30 epochs

---

### What this notebook does
- Loads GT crops from NIH BBBC041 JSON annotations (no watershed at training time)
- Trains EfficientNet-B0 with Focal Loss (γ=2.0, per-class α = inverse frequency)
- Validates every epoch, saves `best.pth` and `last.pth`
- Outputs `metrics.json` and `loss_curves.png`

### Resume guide
Set `RESUME_FROM` to `/kaggle/input/<your-dataset>/last.pth` if continuing from a previous session.

## 1 · Configuration

In [ ]:
from pathlib import Path

# ── UPDATE these paths to match your Kaggle dataset slug ──────────────────
DATASET_ROOT = Path('/kaggle/input/datasets/kaysarulanas/bbbc041-malaria/malaria')
# ──────────────────────────────────────────────────────────────────────────

TRAIN_JSON = DATASET_ROOT / 'training.json'
IMG_DIR    = DATASET_ROOT / 'images'

OUT_DIR    = Path('/kaggle/working/phase3_checkpoints')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Training hyperparameters
EPOCHS      = 30
BATCH_SIZE  = 64
LR          = 1e-4
VAL_FRAC    = 0.15
NUM_WORKERS = 2
CROP_SIZE   = 64
GAMMA       = 2.0    # Focal Loss focusing exponent

# Resume from a previous checkpoint (None = start fresh)
RESUME_FROM = None
# RESUME_FROM = '/kaggle/input/your-phase3-checkpoint/last.pth'

print(f'Train JSON : {TRAIN_JSON}')
print(f'Images     : {IMG_DIR}')
print(f'Output dir : {OUT_DIR}')
assert TRAIN_JSON.exists(), f'training.json not found at {TRAIN_JSON}'
assert IMG_DIR.exists(),    f'images dir not found at {IMG_DIR}'
print('Paths OK.')

## 2 · GPU Check & Dependencies

In [ ]:
import subprocess, sys

# Check GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'No GPU found')

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device  : {torch.cuda.get_device_name(0)}')

# scikit-image for watershed (Stage 1 only — not needed for Stage 2 training)
# Uncomment if running Stage 1 in this notebook:
# subprocess.run([sys.executable, '-m', 'pip', 'install', 'scikit-image', '-q'])

## 3 · Imports

In [ ]:
import json, time, csv
from collections import defaultdict, Counter
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torchvision.transforms as T
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 4 · Label Map

In [ ]:
# Single source of truth — mirrors shared/label_map.py
LABEL_TO_INT = {
    'background':     0,
    'red blood cell': 1,
    'trophozoite':    2,
    'ring':           3,
    'schizont':       4,
    'gametocyte':     5,
    'leukocyte':      6,
}
INT_TO_LABEL   = {v: k for k, v in LABEL_TO_INT.items()}
NUM_CLASSES    = 7
PARASITE_NAMES = ['trophozoite', 'ring', 'schizont', 'gametocyte']
SKIP_LABELS    = {'difficult'}

# Training annotation counts from Phase 1 EDA
TRAIN_COUNTS = {1: 77420, 2: 1473, 3: 353, 4: 179, 5: 144, 6: 103}

print('Label map:')
for idx, name in INT_TO_LABEL.items():
    cnt = TRAIN_COUNTS.get(idx, 0)
    print(f'  [{idx}] {name:<22}  count={cnt:>6}')

## 5 · Dataset (GT Crops)

In [ ]:
class MalariaCropDataset(Dataset):
    """
    Extracts 64x64 crops from GT bounding boxes.
    Used for Stage 2 training — labels come from annotations, not watershed.
    At inference, crops come from Stage 1 watershed and are classified by the
    trained model.
    """

    # ImageNet stats (EfficientNet-B0 pretrained on ImageNet)
    MEAN = [0.485, 0.456, 0.406]
    STD  = [0.229, 0.224, 0.225]

    def __init__(self, json_path, img_dir, train=True, crop_size=CROP_SIZE):
        self.img_dir    = Path(img_dir)
        self.crop_size  = crop_size
        self.items      = []   # list of (img_path, x1, y1, x2, y2, label_idx)

        aug = [
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.RandomRotation(15),
            T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
        ] if train else []

        self.transform = T.Compose([
            T.Resize((crop_size, crop_size)),
            *aug,
            T.ToTensor(),
            T.Normalize(self.MEAN, self.STD),
        ])

        self._parse(json_path)
        print(f'MalariaCropDataset: {len(self.items)} crops '
              f'({"train" if train else "val"} mode)')
        # Class balance
        counts = Counter(it[5] for it in self.items)
        for idx in range(1, NUM_CLASSES):
            c = counts.get(idx, 0)
            print(f'  [{idx}] {INT_TO_LABEL[idx]:<22}: {c}')

    def _parse(self, json_path):
        with open(json_path) as f:
            raw = json.load(f)
        for item in raw:
            fname = item['image']['pathname'].lstrip('/')
            img_path = self.img_dir / fname
            if not img_path.exists():
                img_path = self.img_dir / Path(fname).name
            if not img_path.exists():
                continue
            for ann in item.get('objects', []):
                lbl = ann.get('category', '')
                if lbl in SKIP_LABELS or lbl not in LABEL_TO_INT:
                    continue
                bb = ann.get('bounding_box', {})
                x1 = int(bb.get('minimum', {}).get('c', 0))
                y1 = int(bb.get('minimum', {}).get('r', 0))
                x2 = int(bb.get('maximum', {}).get('c', 0))
                y2 = int(bb.get('maximum', {}).get('r', 0))
                if x2 > x1 and y2 > y1:
                    self.items.append(
                        (str(img_path), x1, y1, x2, y2, LABEL_TO_INT[lbl]))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        img_path, x1, y1, x2, y2, label = self.items[idx]
        img = Image.open(img_path).convert('RGB')
        crop = img.crop((x1, y1, x2, y2))
        return self.transform(crop), label


# Load datasets
print('Loading datasets...')
full_ds = MalariaCropDataset(TRAIN_JSON, IMG_DIR, train=True)

n_val   = max(1, int(len(full_ds) * VAL_FRAC))
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(full_ds, [n_train, n_val],
                                 generator=torch.Generator().manual_seed(42))

# Val split uses no-augmentation transform
val_ds.dataset = MalariaCropDataset(TRAIN_JSON, IMG_DIR, train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'\nTrain batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')

## 6 · Focal Loss

In [ ]:
def compute_focal_alpha():
    """Inverse-frequency alpha. Background (class 0) gets weight 0."""
    total = sum(TRAIN_COUNTS.values())
    alpha = [0.0]  # background
    for i in range(1, NUM_CLASSES):
        cnt = TRAIN_COUNTS.get(i, 1)
        alpha.append(total / (len(TRAIN_COUNTS) * cnt))
    # Normalise
    s = sum(alpha[1:])
    alpha = [a / s * (NUM_CLASSES - 1) for a in alpha]
    return torch.tensor(alpha, dtype=torch.float32)


class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma=2.0):
        super().__init__()
        self.register_buffer('alpha', alpha)
        self.gamma = gamma

    def forward(self, logits, targets):
        ce   = F.cross_entropy(logits, targets, reduction='none')
        pt   = torch.exp(-ce)
        at   = self.alpha[targets]
        loss = at * (1.0 - pt) ** self.gamma * ce
        return loss.mean()


alpha     = compute_focal_alpha().to(device)
criterion = FocalLoss(alpha=alpha, gamma=GAMMA)

print('Focal Loss alpha weights:')
for i in range(NUM_CLASSES):
    print(f'  [{i}] {INT_TO_LABEL[i]:<22} alpha={alpha[i].item():.4f}')

## 7 · Model (EfficientNet-B0)

In [ ]:
def build_model():
    weights = EfficientNet_B0_Weights.IMAGENET1K_V1
    model   = efficientnet_b0(weights=weights)
    in_feat = model.classifier[1].in_features   # 1280
    model.classifier[1] = nn.Linear(in_feat, NUM_CLASSES)
    return model

model     = build_model().to(device)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total_params:,}')
print(f'Trainable params: {trainable_params:,}')

## 8 · Resume (optional)

In [ ]:
start_epoch   = 0
best_val_acc  = 0.0
train_losses, val_losses   = [], []
train_accs,   val_accs     = [], []

if RESUME_FROM and Path(RESUME_FROM).exists():
    print(f'Resuming from {RESUME_FROM}')
    ckpt = torch.load(RESUME_FROM, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch   = ckpt['epoch']
    best_val_acc  = ckpt.get('best_val_acc', 0.0)
    train_losses  = ckpt.get('train_losses', [])
    val_losses    = ckpt.get('val_losses',   [])
    train_accs    = ckpt.get('train_accs',   [])
    val_accs      = ckpt.get('val_accs',     [])
    # Fast-forward scheduler
    for _ in range(start_epoch):
        scheduler.step()
    print(f'  Resuming from epoch {start_epoch + 1}  '
          f'(best val acc so far: {best_val_acc*100:.2f}%)')
else:
    print('Starting fresh (no resume checkpoint).')

## 9 · Training & Validation Functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for crops, labels in loader:
        crops, labels = crops.to(device), labels.to(device)
        logits = model(crops)
        loss   = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += len(labels)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    class_correct = defaultdict(int)
    class_total   = defaultdict(int)
    for crops, labels in loader:
        crops, labels = crops.to(device), labels.to(device)
        logits  = model(crops)
        loss    = criterion(logits, labels)
        preds   = logits.argmax(1)
        total_loss += loss.item() * len(labels)
        correct    += (preds == labels).sum().item()
        total      += len(labels)
        for lbl, pred in zip(labels.cpu(), preds.cpu()):
            class_total[lbl.item()]   += 1
            class_correct[lbl.item()] += int(pred == lbl)

    per_class = {}
    for i in range(1, NUM_CLASSES):
        if class_total[i] > 0:
            per_class[INT_TO_LABEL[i]] = round(
                class_correct[i] / class_total[i] * 100, 2)

    return total_loss / total, correct / total, per_class

print('Training functions defined.')

## 10 · Training Loop

In [ ]:
print(f'Training epochs {start_epoch+1} → {EPOCHS}')
print(f'Batch size: {BATCH_SIZE}  |  LR: {LR}  |  Focal γ: {GAMMA}')
print('-' * 80)

for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()

    tr_loss, tr_acc          = train_one_epoch(model, train_loader, criterion, optimizer)
    vl_loss, vl_acc, per_cls = evaluate(model, val_loader, criterion)
    scheduler.step()

    train_losses.append(tr_loss); val_losses.append(vl_loss)
    train_accs.append(tr_acc);    val_accs.append(vl_acc)

    elapsed = time.time() - t0
    print(f'Epoch [{epoch+1:03d}/{EPOCHS}] '
          f'train_loss={tr_loss:.4f}  val_loss={vl_loss:.4f}  '
          f'train_acc={tr_acc*100:.2f}%  val_acc={vl_acc*100:.2f}%  '
          f'lr={scheduler.get_last_lr()[0]:.2e}  ({elapsed:.0f}s)')

    is_best = vl_acc > best_val_acc
    if is_best:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), OUT_DIR / 'best.pth')
        print(f'  *** New best val acc: {best_val_acc*100:.2f}% — saved best.pth ***')
        print(f'  Per-class: ' +
              '  '.join(f"{k[:5]}={v}%" for k, v in per_cls.items()))

    # Always save last checkpoint
    torch.save({
        'epoch':        epoch + 1,
        'model':        model.state_dict(),
        'optimizer':    optimizer.state_dict(),
        'scheduler':    scheduler.state_dict(),
        'best_val_acc': best_val_acc,
        'train_losses': train_losses,
        'val_losses':   val_losses,
        'train_accs':   train_accs,
        'val_accs':     val_accs,
        'cfg': {
            'epochs': EPOCHS, 'batch': BATCH_SIZE,
            'lr': LR, 'gamma': GAMMA,
        }
    }, OUT_DIR / 'last.pth')

print('\nTraining complete.')

## 11 · Final Evaluation (best checkpoint)

In [ ]:
print('Loading best.pth for final evaluation...')
model.load_state_dict(torch.load(OUT_DIR / 'best.pth', map_location=device))
_, final_acc, final_per_class = evaluate(model, val_loader, criterion)

print(f'\nFinal Val Accuracy (best ckpt): {final_acc*100:.2f}%')
print('Per-class accuracy:')
for cls, acc in sorted(final_per_class.items(), key=lambda x: -x[1]):
    print(f'  {cls:<22}: {acc:.2f}%')

# Save metrics
metrics = {
    'val_accuracy':       round(final_acc * 100, 2),
    'best_val_acc_pct':   round(best_val_acc * 100, 2),
    'total_epochs':       EPOCHS,
    'per_class_accuracy': final_per_class,
    'train_losses':       [round(x, 6) for x in train_losses],
    'val_losses':         [round(x, 6) for x in val_losses],
    'train_accs':         [round(x, 4) for x in train_accs],
    'val_accs':           [round(x, 4) for x in val_accs],
}
metrics_path = OUT_DIR / 'metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'\nMetrics saved: {metrics_path}')

## 12 · Loss & Accuracy Curves

In [ ]:
epochs_x = list(range(1, len(train_losses) + 1))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(epochs_x, train_losses, 'b-o', ms=3, label='Train loss')
axes[0].plot(epochs_x, val_losses,   'r-o', ms=3, label='Val loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Focal Loss')
axes[0].set_title('EfficientNet-B0 Stage 2 — Focal Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_x, [a*100 for a in train_accs], 'b-o', ms=3, label='Train acc')
axes[1].plot(epochs_x, [a*100 for a in val_accs],   'r-o', ms=3, label='Val acc')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('EfficientNet-B0 Stage 2 — Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
curve_path = OUT_DIR / 'loss_curves.png'
plt.savefig(str(curve_path), dpi=120)
plt.show()
print(f'Loss curves saved: {curve_path}')

## 13 · Download Checkpoints

After the run completes, download these files from the Kaggle output panel:

| File | Purpose |
|------|---------|
| `phase3_checkpoints/best.pth` | Best val-accuracy checkpoint — use for inference |
| `phase3_checkpoints/last.pth` | Last epoch checkpoint — use for resuming |
| `phase3_checkpoints/metrics.json` | Accuracy + per-class stats |
| `phase3_checkpoints/loss_curves.png` | Training curves |

Move downloaded files to:  
`Phase3-PipelineB/checkpoints/` in your local repo.

In [ ]:
# List output files
import os
print('Output files:')
for f in sorted(Path('/kaggle/working/phase3_checkpoints').glob('*')):
    size = f.stat().st_size / 1024 / 1024
    print(f'  {f.name:<30} {size:.1f} MB')